In [ ]:
# Setup
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from data import (ensure_dirs, get_pdf_files, extract_text_from_pdf,
                  chunk_text, seed_default_pdfs, DOCS_DIR,
                  DEFAULT_CHUNK_SIZE, DEFAULT_CHUNK_OVERLAP)
from utils import save_fig, plot_chunk_length_distribution

ensure_dirs()
seed_default_pdfs()
print(f"PDF directory: {DOCS_DIR}")

In [ ]:
# 1. List PDFs (or placeholder) in the corpus
pdf_files = get_pdf_files()
print(f"Found {len(pdf_files)} file(s):")
for p in pdf_files:
    print(f"  - {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
# 2. Extract text + chunk each document
rows = []
all_chunks = []
for pdf_path in pdf_files:
    try:
        text = extract_text_from_pdf(pdf_path)
    except RuntimeError:
        # Plain-text fallback for the bundled placeholder
        text = pdf_path.read_text(errors='ignore')
    chunks = chunk_text(text, chunk_size=DEFAULT_CHUNK_SIZE, overlap=DEFAULT_CHUNK_OVERLAP)
    rows.append({
        'file': pdf_path.name,
        'text_chars': len(text),
        'n_chunks': len(chunks),
        'avg_chunk_len': (sum(len(c) for c in chunks) / len(chunks)) if chunks else 0,
    })
    all_chunks.extend(chunks)

print(pd.DataFrame(rows).to_string(index=False))
print(f"\nTotal chunks across corpus: {len(all_chunks)}")

In [ ]:
# 3. Chunk-length distribution
plot_chunk_length_distribution(all_chunks)
save_fig('01_chunk_length_distribution.png')
plt.show()

In [ ]:
# 4. Inspect a sample chunk
sample = all_chunks[0] if all_chunks else ""
print("Sample chunk (first 500 chars):")
print(sample[:500])
print(f"\nLength: {len(sample)} chars")